In [0]:
%pip install pyproj geojson httpx rasterio

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
from datetime import datetime, timezone, timedelta
import time

from pyspark.sql import functions as F
import httpx
import numpy as np
import pandas as pd
import pyproj
import rasterio
from geojson import Feature, Polygon
from io import BytesIO
import requests as req

from pyspark.sql import functions as F
from pyspark.sql.types import FloatType
from pyspark.sql.window import Window

from delta.tables import DeltaTable


CATALOG = "main"
SCHEMA  = "default"
VOLUME  = "crop_health"
 
VOL_BASE = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}"
 
INPUT_SENTINEL2  = f"{VOL_BASE}/sentinel2_manifest.parquet"
INPUT_WEATHER    = f"{VOL_BASE}/weather.parquet"
OUTPUT_PROCESSED = f"{VOL_BASE}/processed_delta"   # Delta table folder
OUTPUT_PARQUET   = f"{VOL_BASE}/processed.parquet" # flat parquet for Airflow


STAC_URL = "https://earth-search.aws.element84.com/v1/search"

BBOXES = [
    (-91.5, 32.5, -90.5, 33.0),   # AOI-1
    (-90.8, 31.5, -90.0, 32.5),   # AOI-2
]

# Sample points per AOI — keeps data size manageable
SAMPLE_INTERVAL = 10  # every 10th pixel



In [0]:
# Timezone-aware UTC helpers (fixes deprecation warning)
def utc_today() -> str:
    return datetime.now(timezone.utc).strftime("%Y-%m-%d")
 
def utc_days_ago(n: int) -> str:
    return (datetime.now(timezone.utc) - timedelta(days=n)).strftime("%Y-%m-%d")
 
print(f"Volume base path : {VOL_BASE}")
print(f"Today (UTC)      : {utc_today()}")

Volume base path : /Volumes/main/default/crop_health
Today (UTC)      : 2026-04-25


In [0]:
# Cell 3 — Ingest + Real Pixel Sampling from Sentinel-2 COGs

def query_stac(bbox: tuple, days_back: int = 14) -> list:
    lon_min, lat_min, lon_max, lat_max = bbox
    geometry = Feature(geometry=Polygon([[
        (lon_min, lat_min), (lon_max, lat_min),
        (lon_max, lat_max), (lon_min, lat_max),
        (lon_min, lat_min),
    ]])).geometry

    all_features, page = [], 1
    while True:
        resp = httpx.post(STAC_URL, json={
            "datetime":    f"{utc_days_ago(days_back)}T00:00:00Z/{utc_today()}T23:59:59Z",
            "intersects":  geometry,
            "collections": ["sentinel-2-l2a"],
            "limit":       10,   # fewer scenes — each one takes time to sample
            "page":        page,
            "query":       {"eo:cloud_cover": {"lt": 20}},
        }, timeout=30)
        features = resp.json().get("features", [])
        if not features:
            break
        all_features.extend(features)
        page += 1
    return all_features


def sample_band(lat_arr, lon_arr, band_url: str) -> list:
    """
    Download a single Sentinel-2 band (Cloud Optimised GeoTIFF)
    and sample it at the given lat/lon coordinates.
    Returns a list of integer reflectance values.
    """
    try:
        resp = req.get(band_url, stream=True, timeout=60)
        resp.raise_for_status()
        with rasterio.open(BytesIO(resp.content)) as src:
            # Transform lat/lon (EPSG:4326) → band CRS
            transformer = pyproj.Transformer.from_crs(
                "EPSG:4326", src.crs, always_xy=True
            )
            xs, ys = transformer.transform(lon_arr, lat_arr)
            # Sample raster at each point
            samples = [
                int(list(src.sample([(x, y)]))[0][0])
                for x, y in zip(xs, ys)
            ]
        return samples
    except Exception as e:
        print(f"  Band sampling failed ({band_url[:60]}...): {e}")
        return [0] * len(lat_arr)


def build_sample_grid_from_scene(feature: dict, n: int = 25):
    """Build a sample grid from the actual scene bounding box."""
    bbox = feature.get("bbox")  # [lon_min, lat_min, lon_max, lat_max]
    if not bbox:
        return None, None
    lon_min, lat_min, lon_max, lat_max = bbox
    lons = np.linspace(lon_min + 0.01, lon_max - 0.01, n)
    lats = np.linspace(lat_min + 0.01, lat_max - 0.01, n)
    lon_grid, lat_grid = np.meshgrid(lons, lats)
    return lat_grid.ravel(), lon_grid.ravel()


rows = []
for bbox in BBOXES:
    features = query_stac(bbox, days_back=14)
    print(f"AOI {bbox} → {len(features)} scenes")

    for f in features:
        props  = f.get("properties", {})
        assets = f.get("assets", {})
        scene_date = props.get("datetime", "")[:10]

        nir_url = assets.get("nir", {}).get("href")
        red_url = assets.get("red", {}).get("href")

        if not nir_url or not red_url:
            print(f"  Skipping {f['id']} — missing band URLs")
            continue

        # Build grid from THIS scene's actual bbox, not the AOI bbox
        lat_arr, lon_arr = build_sample_grid_from_scene(f, n=25)
        if lat_arr is None:
            print(f"  Skipping {f['id']} — no bbox")
            continue

        print(f"  Sampling scene {f['id']} ({scene_date}, {len(lat_arr)} points)...")

        nir_vals = sample_band(lat_arr, lon_arr, nir_url)
        red_vals = sample_band(lat_arr, lon_arr, red_url)

        for i in range(len(lat_arr)):
            rows.append({
                "scene_id":    f["id"],
                "datetime":    scene_date,
                "cloud_cover": props.get("eo:cloud_cover"),
                "aoi_bbox":    str(bbox),
                "nir_href":    nir_url,
                "red_href":    red_url,
                "nir":         nir_vals[i],
                "red":         red_vals[i],
                "lat":         float(lat_arr[i]),
                "lon":         float(lon_arr[i]),
            })

print(f"\nTotal rows collected: {len(rows)}")
sat_pdf = pd.DataFrame(rows)

print(f"\nRows before nodata filter: {len(sat_pdf)}")
print(sat_pdf.groupby("datetime")[["nir","red"]].mean())
print(f"\nZero-value rows by date:")
print(sat_pdf[sat_pdf["nir"] == 0].groupby("datetime").size())

# Filter out zero-value pixels (nodata / outside scene extent)
sat_pdf = sat_pdf[(sat_pdf["nir"] > 0) & (sat_pdf["red"] > 0)]
print(f"After nodata filter: {len(sat_pdf)} rows")

sat_df = spark.createDataFrame(sat_pdf)
sat_df.write.mode("append").parquet(INPUT_SENTINEL2)
print(f"\nWrote {len(sat_pdf)} rows → {INPUT_SENTINEL2}")
display(sat_df.limit(10))

AOI (-91.5, 32.5, -90.5, 33.0) → 0 scenes
AOI (-90.8, 31.5, -90.0, 32.5) → 10 scenes
  Sampling scene S2A_15RYQ_20260423_0_L2A (2026-04-23, 625 points)...
  Sampling scene S2A_16RBV_20260423_0_L2A (2026-04-23, 625 points)...
  Sampling scene S2A_15SYR_20260423_0_L2A (2026-04-23, 625 points)...
  Sampling scene S2A_16SBA_20260423_0_L2A (2026-04-23, 625 points)...
  Sampling scene S2C_16RBV_20260414_0_L2A (2026-04-14, 625 points)...
  Sampling scene S2C_16SBA_20260414_0_L2A (2026-04-14, 625 points)...
  Sampling scene S2C_16SBB_20260414_0_L2A (2026-04-14, 625 points)...
  Sampling scene S2C_15RYQ_20260411_0_L2A (2026-04-11, 625 points)...
  Sampling scene S2C_16RBV_20260411_0_L2A (2026-04-11, 625 points)...
  Sampling scene S2C_15SYR_20260411_0_L2A (2026-04-11, 625 points)...

Total rows collected: 6250

Rows before nodata filter: 6250
                    nir         red
datetime                           
2026-04-11  2272.397867  402.195733
2026-04-14  2535.545067  599.181333
2026-04-23

scene_id,datetime,cloud_cover,aoi_bbox,nir_href,red_href,nir,red,lat,lon
S2A_15RYQ_20260423_0_L2A,2026-04-23,0.843252,"(-90.8, 31.5, -90.0, 32.5)",https://sentinel-cogs.s3.us-west-2.amazonaws.com/sentinel-s2-l2a-cogs/15/R/YQ/2026/4/S2A_15RYQ_20260423_0_L2A/B08.tif,https://sentinel-cogs.s3.us-west-2.amazonaws.com/sentinel-s2-l2a-cogs/15/R/YQ/2026/4/S2A_15RYQ_20260423_0_L2A/B04.tif,3809,308,30.614681,-90.19560208333333
S2A_15RYQ_20260423_0_L2A,2026-04-23,0.843252,"(-90.8, 31.5, -90.0, 32.5)",https://sentinel-cogs.s3.us-west-2.amazonaws.com/sentinel-s2-l2a-cogs/15/R/YQ/2026/4/S2A_15RYQ_20260423_0_L2A/B08.tif,https://sentinel-cogs.s3.us-west-2.amazonaws.com/sentinel-s2-l2a-cogs/15/R/YQ/2026/4/S2A_15RYQ_20260423_0_L2A/B04.tif,3848,315,30.614681,-90.175151625
S2A_15RYQ_20260423_0_L2A,2026-04-23,0.843252,"(-90.8, 31.5, -90.0, 32.5)",https://sentinel-cogs.s3.us-west-2.amazonaws.com/sentinel-s2-l2a-cogs/15/R/YQ/2026/4/S2A_15RYQ_20260423_0_L2A/B08.tif,https://sentinel-cogs.s3.us-west-2.amazonaws.com/sentinel-s2-l2a-cogs/15/R/YQ/2026/4/S2A_15RYQ_20260423_0_L2A/B04.tif,2455,349,30.614681,-90.15470116666667
S2A_15RYQ_20260423_0_L2A,2026-04-23,0.843252,"(-90.8, 31.5, -90.0, 32.5)",https://sentinel-cogs.s3.us-west-2.amazonaws.com/sentinel-s2-l2a-cogs/15/R/YQ/2026/4/S2A_15RYQ_20260423_0_L2A/B08.tif,https://sentinel-cogs.s3.us-west-2.amazonaws.com/sentinel-s2-l2a-cogs/15/R/YQ/2026/4/S2A_15RYQ_20260423_0_L2A/B04.tif,3569,292,30.614681,-90.13425070833334
S2A_15RYQ_20260423_0_L2A,2026-04-23,0.843252,"(-90.8, 31.5, -90.0, 32.5)",https://sentinel-cogs.s3.us-west-2.amazonaws.com/sentinel-s2-l2a-cogs/15/R/YQ/2026/4/S2A_15RYQ_20260423_0_L2A/B08.tif,https://sentinel-cogs.s3.us-west-2.amazonaws.com/sentinel-s2-l2a-cogs/15/R/YQ/2026/4/S2A_15RYQ_20260423_0_L2A/B04.tif,4285,1078,30.614681,-90.11380025
S2A_15RYQ_20260423_0_L2A,2026-04-23,0.843252,"(-90.8, 31.5, -90.0, 32.5)",https://sentinel-cogs.s3.us-west-2.amazonaws.com/sentinel-s2-l2a-cogs/15/R/YQ/2026/4/S2A_15RYQ_20260423_0_L2A/B08.tif,https://sentinel-cogs.s3.us-west-2.amazonaws.com/sentinel-s2-l2a-cogs/15/R/YQ/2026/4/S2A_15RYQ_20260423_0_L2A/B04.tif,3557,544,30.614681,-90.09334979166667
S2A_15RYQ_20260423_0_L2A,2026-04-23,0.843252,"(-90.8, 31.5, -90.0, 32.5)",https://sentinel-cogs.s3.us-west-2.amazonaws.com/sentinel-s2-l2a-cogs/15/R/YQ/2026/4/S2A_15RYQ_20260423_0_L2A/B08.tif,https://sentinel-cogs.s3.us-west-2.amazonaws.com/sentinel-s2-l2a-cogs/15/R/YQ/2026/4/S2A_15RYQ_20260423_0_L2A/B04.tif,2339,925,30.614681,-90.07289933333334
S2A_15RYQ_20260423_0_L2A,2026-04-23,0.843252,"(-90.8, 31.5, -90.0, 32.5)",https://sentinel-cogs.s3.us-west-2.amazonaws.com/sentinel-s2-l2a-cogs/15/R/YQ/2026/4/S2A_15RYQ_20260423_0_L2A/B08.tif,https://sentinel-cogs.s3.us-west-2.amazonaws.com/sentinel-s2-l2a-cogs/15/R/YQ/2026/4/S2A_15RYQ_20260423_0_L2A/B04.tif,4702,3049,30.614681,-90.052448875
S2A_15RYQ_20260423_0_L2A,2026-04-23,0.843252,"(-90.8, 31.5, -90.0, 32.5)",https://sentinel-cogs.s3.us-west-2.amazonaws.com/sentinel-s2-l2a-cogs/15/R/YQ/2026/4/S2A_15RYQ_20260423_0_L2A/B08.tif,https://sentinel-cogs.s3.us-west-2.amazonaws.com/sentinel-s2-l2a-cogs/15/R/YQ/2026/4/S2A_15RYQ_20260423_0_L2A/B04.tif,3289,510,30.614681,-90.03199841666667
S2A_15RYQ_20260423_0_L2A,2026-04-23,0.843252,"(-90.8, 31.5, -90.0, 32.5)",https://sentinel-cogs.s3.us-west-2.amazonaws.com/sentinel-s2-l2a-cogs/15/R/YQ/2026/4/S2A_15RYQ_20260423_0_L2A/B08.tif,https://sentinel-cogs.s3.us-west-2.amazonaws.com/sentinel-s2-l2a-cogs/15/R/YQ/2026/4/S2A_15RYQ_20260423_0_L2A/B04.tif,3684,342,30.614681,-90.01154795833334


In [0]:
# Fetch weather from Open-Meteo and write to Volume
#  
LOCATIONS = [
    {"name": "AOI-1", "lat": 32.75, "lon": -91.0},
    {"name": "AOI-2", "lat": 32.0,  "lon": -90.4},
]
HOURLY_VARS = "temperature_2m,precipitation,shortwave_radiation,relative_humidity_2m"
 
wx_rows = []
for loc in LOCATIONS:
    resp = httpx.get("https://api.open-meteo.com/v1/forecast", params={
        "latitude":   loc["lat"],
        "longitude":  loc["lon"],
        "hourly":     HOURLY_VARS,
        "start_date": utc_days_ago(14),
        "end_date":   utc_today(),
        "timezone":   "UTC",
    }, timeout=30)
    hourly = resp.json().get("hourly", {})
    df = pd.DataFrame(hourly).rename(columns={"time": "datetime"})
    df["location_name"] = loc["name"]
    df["lat"] = loc["lat"]
    df["lon"] = loc["lon"]
    wx_rows.append(df)
    print(f"  {loc['name']} -> {len(df)} hourly rows")
 
wx_pdf = pd.concat(wx_rows, ignore_index=True)
wx_df  = spark.createDataFrame(wx_pdf)
wx_df.write.mode("append").parquet(INPUT_WEATHER)
print(f"\nWrote {len(wx_pdf)} weather rows -> {INPUT_WEATHER}")
display(wx_df.limit(5))

  AOI-1 -> 360 hourly rows
  AOI-2 -> 360 hourly rows

Wrote 720 weather rows -> /Volumes/main/default/crop_health/weather.parquet


datetime,temperature_2m,precipitation,shortwave_radiation,relative_humidity_2m,location_name,lat,lon
2026-04-11T00:00,26.0,0.0,113.0,40,AOI-1,32.75,-91.0
2026-04-11T01:00,22.2,0.0,0.0,49,AOI-1,32.75,-91.0
2026-04-11T02:00,21.1,0.0,0.0,48,AOI-1,32.75,-91.0
2026-04-11T03:00,19.9,0.0,0.0,50,AOI-1,32.75,-91.0
2026-04-11T04:00,18.8,0.0,0.0,59,AOI-1,32.75,-91.0


In [0]:
sat_df = spark.read.parquet(INPUT_SENTINEL2)
sat_df = sat_df.dropDuplicates(["scene_id", "datetime", "lat", "lon"]) # deduplicate across runs
 
@F.udf(FloatType())
def ndvi_udf(nir, red):
    """T1 - Normalised Difference Vegetation Index. Returns None on divide-by-zero."""
    if nir is None or red is None:
        return None
    denom = float(nir + red)
    return None if denom == 0.0 else float((nir - red) / denom)
 
sat_df = sat_df.withColumn("ndvi", ndvi_udf(F.col("nir"), F.col("red")))
 
sat_df = sat_df.withColumn(
    "ndvi_class",
    F.when(F.col("ndvi") < 0.0,  "Water/Non-Vegetated")
     .when(F.col("ndvi") < 0.2,  "Sparse Vegetation")
     .when(F.col("ndvi") < 0.4,  "Moderate Vegetation")
     .when(F.col("ndvi") < 0.6,  "Dense Vegetation")
     .otherwise("Very Dense Vegetation"),
)
 
print("T1 NDVI computed")
display(sat_df.select("scene_id", "datetime", "nir", "red", "ndvi", "ndvi_class").limit(20))

T1 NDVI computed


scene_id,datetime,nir,red,ndvi,ndvi_class
S2C_16RBV_20260414_0_L2A,2026-04-14,2822,241,0.84263796,Very Dense Vegetation
S2C_16RBV_20260414_0_L2A,2026-04-14,1157,104,0.83505154,Very Dense Vegetation
S2C_16SBA_20260414_0_L2A,2026-04-14,2964,743,0.59913677,Dense Vegetation
S2C_16RBV_20260414_0_L2A,2026-04-14,1063,555,0.31396785,Moderate Vegetation
S2C_16SBA_20260414_0_L2A,2026-04-14,451,593,-0.13601533,Water/Non-Vegetated
S2C_16SBB_20260414_0_L2A,2026-04-14,3224,1245,0.4428284,Dense Vegetation
S2C_16SBB_20260414_0_L2A,2026-04-14,3169,280,0.8376341,Very Dense Vegetation
S2C_16SBB_20260414_0_L2A,2026-04-14,2808,1591,0.2766538,Moderate Vegetation
S2C_16SBB_20260414_0_L2A,2026-04-14,406,503,-0.10671067,Water/Non-Vegetated
S2C_16RBV_20260414_0_L2A,2026-04-14,4509,287,0.8803169,Very Dense Vegetation


In [0]:
wx_df = spark.read.parquet(INPUT_WEATHER)
wx_df = wx_df.dropDuplicates(["datetime", "location_name"])  # deduplicate across runs
 
# Aggregate hourly -> daily averages/sums
daily_wx = (
    wx_df
    .withColumn("date", F.to_date("datetime"))
    .groupBy("date", "location_name")
    .agg(
        F.avg("temperature_2m").alias("avg_temp_c"),
        F.sum("precipitation").alias("total_precip_mm"),
        F.avg("shortwave_radiation").alias("avg_solar_radiation"),
        F.avg("relative_humidity_2m").alias("avg_humidity_pct"),
    )
)
 
# Left-join satellite scenes with daily weather on date
enriched_df = (
    sat_df
    .withColumn("scene_date_dt", F.to_date("datetime"))
    .join(daily_wx, F.col("scene_date_dt") == daily_wx["date"], how="left")
    .drop("date", "scene_date_dt")
)
 
print(f"T2 Weather enrichment done — {enriched_df.count()} rows")
display(enriched_df.select(
    "scene_id", "datetime", "ndvi",
    "avg_temp_c", "total_precip_mm", "avg_solar_radiation"
).limit(20))

T2 Weather enrichment done — 284 rows


scene_id,datetime,ndvi,avg_temp_c,total_precip_mm,avg_solar_radiation
S2C_16RBV_20260414_0_L2A,2026-04-14,0.84263796,22.6375,0.0,294.0833333333333
S2C_16RBV_20260414_0_L2A,2026-04-14,0.83505154,22.6375,0.0,294.0833333333333
S2C_16SBA_20260414_0_L2A,2026-04-14,0.59913677,22.6375,0.0,294.0833333333333
S2C_16RBV_20260414_0_L2A,2026-04-14,0.31396785,22.6375,0.0,294.0833333333333
S2C_16SBA_20260414_0_L2A,2026-04-14,-0.13601533,22.6375,0.0,294.0833333333333
S2C_16SBB_20260414_0_L2A,2026-04-14,0.4428284,22.6375,0.0,294.0833333333333
S2C_16SBB_20260414_0_L2A,2026-04-14,0.8376341,22.6375,0.0,294.0833333333333
S2C_16SBB_20260414_0_L2A,2026-04-14,0.2766538,22.6375,0.0,294.0833333333333
S2C_16SBB_20260414_0_L2A,2026-04-14,-0.10671067,22.6375,0.0,294.0833333333333
S2C_16RBV_20260414_0_L2A,2026-04-14,0.8803169,22.6375,0.0,294.0833333333333


In [0]:

# Partition by pixel location, order by timestamp
w = (
    Window
    .partitionBy("lat", "lon")
    .orderBy(F.unix_timestamp(F.to_date("datetime")))
    .rangeBetween(-30 * 86_400, 0)   # 30-day trailing window in seconds
)
 
final_df = (
    enriched_df
    .withColumn("rolling_mean_ndvi", F.avg("ndvi").over(w))
    .withColumn(
        "ndvi_drop_pct",
        F.when(
            F.col("rolling_mean_ndvi").isNotNull() & (F.col("rolling_mean_ndvi") > 0),
            (F.col("rolling_mean_ndvi") - F.col("ndvi")) / F.col("rolling_mean_ndvi"),
        ).otherwise(None),
    )
    .withColumn("is_anomaly", F.col("ndvi_drop_pct") > 0.20)
)
 
anomaly_count = final_df.filter(F.col("is_anomaly") == True).count()
print(f"T3 Anomaly flagging done — {anomaly_count} anomalies detected")
display(final_df.select(
    "scene_id", "datetime", "ndvi",
    "rolling_mean_ndvi", "ndvi_drop_pct", "is_anomaly",
).limit(20))

T3 Anomaly flagging done — 0 anomalies detected


scene_id,datetime,ndvi,rolling_mean_ndvi,ndvi_drop_pct,is_anomaly
S2C_16RBV_20260414_0_L2A,2026-04-14,0.37525612,0.37525612115859985,0.0,false
S2C_16RBV_20260414_0_L2A,2026-04-14,0.37525612,0.37525612115859985,0.0,false
S2C_16RBV_20260414_0_L2A,2026-04-14,0.14145833,0.14145833253860474,0.0,false
S2C_16RBV_20260414_0_L2A,2026-04-14,0.14145833,0.14145833253860474,0.0,false
S2C_16RBV_20260414_0_L2A,2026-04-14,0.69381315,0.693813145160675,0.0,false
S2C_16RBV_20260414_0_L2A,2026-04-14,0.69381315,0.693813145160675,0.0,false
S2C_16RBV_20260414_0_L2A,2026-04-14,0.71428573,0.7142857313156128,0.0,false
S2C_16RBV_20260414_0_L2A,2026-04-14,0.71428573,0.7142857313156128,0.0,false
S2C_16RBV_20260414_0_L2A,2026-04-14,0.31396785,0.31499259173870087,0.0032532128723515223,false
S2C_16SBA_20260414_0_L2A,2026-04-14,0.31601733,0.31499259173870087,-0.0032532128723515223,false


In [0]:

if DeltaTable.isDeltaTable(spark, OUTPUT_PROCESSED):
    dt = DeltaTable.forPath(spark, OUTPUT_PROCESSED)
    dt.alias("t").merge(
        final_df.alias("s"),
        """t.scene_id = s.scene_id 
           AND t.datetime = s.datetime 
           AND t.lat = s.lat 
           AND t.lon = s.lon"""
    ).whenNotMatchedInsertAll().execute()
    print("Merged new rows into Delta table")
else:
    final_df.write.format("delta").mode("overwrite").save(OUTPUT_PROCESSED)
    print("Created Delta table")

full_df = spark.read.format("delta").load(OUTPUT_PROCESSED)
full_df = full_df.dropDuplicates(["scene_id", "datetime", "lat", "lon"])
print(f"Total rows in Delta table (all history): {full_df.count()}")

Created Delta table
Total rows in Delta table (all history): 142


In [0]:

final_path = f"{VOL_BASE}/processed.parquet"

# ─── Clean up anything at that path ───
try:
    dbutils.fs.rm(final_path, recurse=True)
    time.sleep(2)
except Exception:
    pass

# ─── Write full history snapshot for Airflow ───
full_df.coalesce(1).write \
    .mode("overwrite") \
    .parquet(final_path)

time.sleep(10)

# ─── Verify it's a directory containing a real part file ───
contents = dbutils.fs.ls(final_path)
part_files = [f for f in contents if f.name.endswith(".parquet") and not f.name.startswith("_")]

if not part_files:
    raise RuntimeError(f"No part file found in {final_path}. Contents: {[f.name for f in contents]}")

print(f"✅ verified — {part_files[0].name} ({part_files[0].size:,} bytes)")
display(contents)

✅ verified — part-00000-tid-3936403602721756657-a94fa9ce-caad-4c33-b5cc-1762b50d42a3-2313-1.c000.snappy.parquet (10,176 bytes)


path,name,size,modificationTime
dbfs:/Volumes/main/default/crop_health/processed.parquet/_SUCCESS,_SUCCESS,0,1777136739000
dbfs:/Volumes/main/default/crop_health/processed.parquet/_committed_3936403602721756657,_committed_3936403602721756657,125,1777136739000
dbfs:/Volumes/main/default/crop_health/processed.parquet/_started_3936403602721756657,_started_3936403602721756657,0,1777136739000
dbfs:/Volumes/main/default/crop_health/processed.parquet/part-00000-tid-3936403602721756657-a94fa9ce-caad-4c33-b5cc-1762b50d42a3-2313-1.c000.snappy.parquet,part-00000-tid-3936403602721756657-a94fa9ce-caad-4c33-b5cc-1762b50d42a3-2313-1.c000.snappy.parquet,10176,1777136739000


In [0]:
total     = full_df.count()
anomalies = full_df.filter("is_anomaly == true").count()
aois      = full_df.select("aoi_bbox").distinct().count()
date_range = full_df.agg(
    F.min("datetime").alias("from_date"),
    F.max("datetime").alias("to_date"),
).first()
 
print("=" * 45)
print(f"  Total rows      : {total}")
print(f"  Anomaly rows    : {anomalies}  ({100 * anomalies / max(total, 1):.1f}%)")
print(f"  AOIs covered    : {aois}")
print(f"  Date range      : {date_range['from_date']}  ->  {date_range['to_date']}")
print("=" * 45)
 
display(
    full_df
    .groupBy("ndvi_class")
    .agg(
        F.count("*").alias("count"),
        F.round(F.avg("ndvi"), 4).alias("mean_ndvi"),
        F.round(F.avg("avg_temp_c"), 2).alias("mean_temp_c"),
    )
    .orderBy("mean_ndvi")
)

  Total rows      : 142
  Anomaly rows    : 0  (0.0%)
  AOIs covered    : 1
  Date range      : 2026-04-14  ->  2026-04-14


ndvi_class,count,mean_ndvi,mean_temp_c
Water/Non-Vegetated,6,-0.1184,22.64
Sparse Vegetation,8,0.158,22.64
Moderate Vegetation,22,0.2882,22.64
Dense Vegetation,15,0.5047,22.64
Very Dense Vegetation,91,0.7886,22.64


In [0]:
display(dbutils.fs.ls("/Volumes/main/default/crop_health/processed.parquet"))


path,name,size,modificationTime
dbfs:/Volumes/main/default/crop_health/processed.parquet/_SUCCESS,_SUCCESS,0,1777136739000
dbfs:/Volumes/main/default/crop_health/processed.parquet/_committed_3936403602721756657,_committed_3936403602721756657,125,1777136739000
dbfs:/Volumes/main/default/crop_health/processed.parquet/_started_3936403602721756657,_started_3936403602721756657,0,1777136739000
dbfs:/Volumes/main/default/crop_health/processed.parquet/part-00000-tid-3936403602721756657-a94fa9ce-caad-4c33-b5cc-1762b50d42a3-2313-1.c000.snappy.parquet,part-00000-tid-3936403602721756657-a94fa9ce-caad-4c33-b5cc-1762b50d42a3-2313-1.c000.snappy.parquet,10176,1777136739000
